<a href="https://colab.research.google.com/github/lukatinarelli/Apuntes/blob/master/Cursos/Python%20Ofensivo%20-%20Hack4u/04%20Programaci%C3%B3n%20Orientada%20a%20Objetos%20(POO)/05%20Encapsulamiento%20y%20m%C3%A9todos%20especiales%20%F0%9F%94%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🔐 Encapsulamiento y Métodos Especiales en Python**

El **encapsulamiento** es la técnica de ocultar los detalles internos de un objeto y proteger su integridad. Los **métodos especiales** (o mágicos) nos permiten definir cómo se comportan nuestros objetos en situaciones comunes (al imprimirlos, sumarlos, compararlos).

## **🏷️ 1. Encapsulamiento: Control de visibilidad**

A diferencia de Java o C++, Python no tiene palabras reservadas como `private` o `public` que prohíban el acceso estricto. En su lugar, usa **convenciones de nombres** basadas en guiones bajos.

### **🟢 1. Atributos Públicos (Sin guiones)**

* **Sintaxis:** `self.nombre`

* **Acceso:** Desde cualquier lugar (dentro y fuera de la clase).

* **Uso:** Información que no es sensible y que no rompe el programa si se modifica.

In [1]:
class Persona:
    def __init__(self, nombre):
        self.nombre = nombre  # Público: Cualquiera puede cambiarlo

p = Persona("Ana")
p.nombre = "Hackeado"  # Sin problemas

### **🟡 2. Atributos Protegidos (Un guion bajo `_`)**

* **Sintaxis:** `self._configuracion`

* **Acceso:** Técnicamente público, pero **semánticamente privado**.

* **Significado:** *"Oye, programador, esto es para uso interno. Si lo tocas desde fuera y rompes algo, es culpa tuya"*.

In [2]:
class BaseDeDatos:
    def __init__(self):
        self._conectado = False  # Protegido

bd = BaseDeDatos()
# Python te DEJA hacerlo, pero tu IDE te avisará de que no deberías.
bd._conectado = True

### **🔴 3. Atributos Privados (Doble guion bajo `__`)**

* **Sintaxis:** `self.__token`

* **Acceso:** Python aplica **Name Mangling** (destroza el nombre) para que sea difícil acceder desde fuera.

* **Uso:** Para evitar conflictos de nombres en herencia y ocultar lógica crítica.

In [ ]:
class CuentaBancaria:
    def __init__(self, saldo):
        self.__saldo = saldo  # Privado

cuenta = CuentaBancaria(1000)

print(cuenta.__saldo)  # ❌ AttributeError: 'CuentaBancaria' object has no attribute '__saldo'

### **🕵️‍♂️ El truco del "Name Mangling" (Visión Ofensiva)**

¿Es realmente privado? **No**. Python simplemente le cambia el nombre internamente para evitar accidentes. Un atacante (o tú mismo depurando) puede acceder así:

#### Formato: `_NombreClase__nombreAtributo`

In [ ]:
# Acceso "forzado" a variable privada
print(cuenta._CuentaBancaria__saldo)  # ✅ Salida: 1000

# Incluso podemos modificarlo
cuenta._CuentaBancaria__saldo = 999999

> 👨‍💻 **Nota para Hackers:**
>
> El encapsulamiento en Python **no es seguridad**. No uses atributos privados para guardar contraseñas o keys pensando que son inaccesibles. Cualquier script puede leer la memoria o usar el *name mangling* para extraerlos.

## **🛠️ 2. Métodos Especiales ("Magic Methods")**

También conocidos como **Dunder Methods** (por *Double UNDERscore*), son el mecanismo que usa Python para realizar la **sobrecarga de operadores**.

No se suelen llamar directamente (`obj.__add__(obj2)`). En su lugar, Python los invoca automáticamente cuando usas sintaxis nativa (`obj + obj2` o `print(obj)`).

### **Tabla de métodos comunes**

| Método | Operación que lo activa | Descripción |
| --- | --- | --- |
| `__init__(self, ...)` | `Clase()` | Constructor. Inicializa el objeto. |
| `__str__(self)` | `print(obj)` | Representación "bonita" para el usuario final. |
| `__repr__(self)` | `repr(obj)` | Representación "técnica" para el programador (debugging). |
| `__eq__(self, other)` | `a == b` | Define cuándo dos objetos son iguales. |
| `__lt__`, `__gt__` | `a < b`, `a > b` | Comparaciones de mayor/menor (para ordenar listas, por ejemplo). |
| `__add__`, `__sub__` | `a + b`, `a - b` | Suma y resta de objetos. |
| `__len__` | `len(obj)` | Longitud del objeto. |

### **Ejemplo práctico: Sobrecarga de operadores**

Imagina que estamos creando una herramienta gráfica y queremos sumar coordenadas (Vectores). Sin métodos mágicos, tendrías que crear un método `sumar_vectores(v1, v2)`. Con ellos, usas `+`.

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    # Define qué pasa cuando ponemos un '+' entre dos vectores
    def __add__(self, other):
        # Devuelve un NUEVO vector con las coordenadas sumadas
        return Vector(self.x + other.x, self.y + other.y)

    # Define qué pasa cuando hacemos print(vector)
    def __str__(self):
        return f"Coordenadas [{self.x}, {self.y}]"

    # Define igualdad (si tienen las mismas coordenadas, son iguales)
    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

v1 = Vector(2, 3)
v2 = Vector(1, 5)
v3 = Vector(2, 3)

# Gracias a __add__
resultado = v1 + v2
print(resultado)  # Gracias a __str__

# Gracias a __eq__
print(v1 == v3)   # (son objetos distintos en memoria, pero sus datos son iguales)

> 💡 **Debug Hack:** Si al hacer un `print(objeto)` te sale algo feo como `<__main__.Persona object at 0x7f...>`, es porque te falta definir el método `__str__`.


## **💡 3. Buenas prácticas**

Aunque Python te da libertad, un buen desarrollador sigue estas reglas:

* **Usa atributos privados (`__`)** solo cuando quieras proteger lógica interna crítica o evitar colisiones de nombres en herencia. No abuses de ello.

* **Prefiere métodos públicos** para que otros objetos interactúen con el tuyo.

* **Documenta los métodos especiales**: Si cambias cómo funciona `__add__` o `__str__`, pon un comentario/docstring explicándolo, o confundirás a quien lea tu código.

* **Respeta las convenciones**: Si ves un atributo con un guion bajo (`_config`), no lo toques desde fuera salvo que sepas muy bien lo que haces.

---

# **🧠 Resumen visual**

| Concepto | Qué es | Cómo se indica | Nivel de "Protección" |
| --- | --- | --- | --- |
| **Público** | Accesible por todos | `self.atributo` | 🔓 Abierto |
| **Protegido** | "No tocar" (Aviso) | `self._atributo` | ⚠️ Solo subclases/interno |
| **Privado** | Difícil acceso (Mangling) | `self.__atributo` | 🔐 Oculto |
| **Mágico** | Comportamiento nativo | `__init__`, `__str__` | ✨ Automático |

---

# **📌 En pocas palabras**

* El **encapsulamiento** no es seguridad blindada en Python, es un sistema de organización y prevención de errores.

* Los **métodos especiales** son la clave para que tus objetos se sientan "profesionales" e integrados en el lenguaje (que se puedan sumar, imprimir o comparar).

* Combinar ambos te permite crear clases que son **cajas negras robustas**: fáciles de usar por fuera (`+`, `print`), pero complejas y protegidas por dentro.

---

# **🏁 Fin de la clase**

Ya sabes cómo proteger los datos de tus objetos y cómo hacer que interactúen con los operadores de Python. Ahora nos falta el último eslabón para controlar el acceso a los datos de forma elegante: **Decoradores y Properties**.

🔙 [**Volver al Índice**](https://github.com/lukatinarelli/Apuntes/blob/master/Cursos/Python%20Ofensivo%20-%20Hack4u/00%20%C3%8Dndice.md)